## Imports and Environment Configuration

- **What it does:** Sets up the project’s data directories (data/raw, data/metadata, and data/processed) using pathlib.Path, and creates the processed-data directory if it does not already exist.

- **Why it matters:** Provides a consistent directory structure for separating raw, metadata, and processed datasets, making the data-preparation workflow reproducible and organized.

In [1]:
import re
import string
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# Define dataset paths relative to the notebooks directory
DATA_RAW_DIR = Path("../data/raw")
DATA_META_DIR = Path("../data/metadata")
DATA_PROC_DIR = Path("../data/processed")

# Ensure output directory exists
DATA_PROC_DIR.mkdir(parents=True, exist_ok=True)

print("Environment configured successfully.")

Environment configured successfully.


## Text Normalization & Cleaning Pipeline

- **What it does:** Defines clean_swahili_text() to normalize Swahili, Sheng, and English transcripts by converting text to lowercase, removing punctuation and unsupported characters, and standardizing whitespace.

- **Why it matters:** Creates a consistent text representation for downstream ASR evaluation and intent-classification tasks while preserving the original transcript for reference.

- **Sanity check:** Tests the normalization function on example Swahili, Sheng, and English/code-switched phrases to verify that the expected cleaning operations are applied.

In [7]:
def clean_trancript_text(text: str) -> str:
    """Normalizes Swahili, Sheng, and English text for ASR evaluation and Intent Classification."""
    if not isinstance(text, str) or pd.isna(text):
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove standard punctuation symbols
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Strip non-alphanumeric characters except basic spaces
    text = re.sub(r"[^a-z0-9\s]", "", text)

    # Collapse multiple whitespaces into a single space
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Sanity check test on code-switched text
sample_input = "Tuma KSH 500 kwa mama, haraka!! 0712345678."
sample_slang = "Niaje tumia brathe soo tano kwa M-Pesa!!"

print(f"Original: {sample_input}")
print(f"Cleaned:  {clean_trancript_text(sample_input)}")

print(f"Original: {sample_slang}")
print(f"Cleaned:  {clean_trancript_text(sample_slang)}")


Original: Tuma KSH 500 kwa mama, haraka!! 0712345678.
Cleaned:  tuma ksh 500 kwa mama haraka 0712345678
Original: Niaje tumia brathe soo tano kwa M-Pesa!!
Cleaned:  niaje tumia brathe soo tano kwa mpesa


## Speech Transcript Processing & Filtering

- **What it does:** Loads the raw transcript metadata and creates a working dataset containing records with valid audio files and available transcripts. The original raw files are not modified or deleted.

- **Why it matters:** Prevents corrupted audio and missing transcripts from entering subsequent analysis and ASR evaluation while preserving the original raw dataset for traceability.

- Applies the transcript-normalization function and stores the result in a separate cleaned_transcript column, preserving the original transcript for comparison and traceability.

In [3]:
# Load raw transcripts index created by extract_data.py
raw_transcripts_path = DATA_RAW_DIR / "transcripts.csv"
df_speech = pd.read_csv(raw_transcripts_path)

print(f"Loaded raw speech records: {len(df_speech)}")

# Filter corrupted audio files and empty transcripts
df_speech_clean = df_speech[
    df_speech["is_valid"] & df_speech["transcript"].notna()
].copy()

# Apply text normalization pipeline
df_speech_clean["cleaned_transcript"] = df_speech_clean["transcript"].apply(
    clean_swahili_text
)

# Export processed speech evaluation dataset
cleaned_speech_path = DATA_PROC_DIR / "cleaned_speech_transcripts.csv"
df_speech_clean.to_csv(cleaned_speech_path, index=False)

print(
    f"Retained {len(df_speech_clean)} / {len(df_speech)} valid speech entries."
)
print(f"Saved to: {cleaned_speech_path}")
df_speech_clean[["file_id", "transcript", "cleaned_transcript"]].head()

Loaded raw speech records: 12171
Retained 12171 / 12171 valid speech entries.
Saved to: ..\data\processed\cleaned_speech_transcripts.csv


,file_id,transcript,cleaned_transcript
0,16k-emission_swahili_05h30_-_06h00_tu_20101124...,ya redio france internanational mimi ni zuhra ...,ya redio france internanational mimi ni zuhra ...
1,16k-emission_swahili_05h30_-_06h00_tu_20101124...,marekani yasema iko tayari kuisaidia korea kus...,marekani yasema iko tayari kuisaidia korea kus...
2,16k-emission_swahili_05h30_-_06h00_tu_20101124...,na tume ya uchaguzi nchini nigeria yataja tare...,na tume ya uchaguzi nchini nigeria yataja tare...
3,16k-emission_swahili_05h30_-_06h00_tu_20101124...,karibu katika awamu ya pili ya matangazo ya as...,karibu katika awamu ya pili ya matangazo ya as...
4,16k-emission_swahili_05h30_-_06h00_tu_20101124...,wachimbaji wote ishirini na kenda ambao waliku...,wachimbaji wote ishirini na kenda ambao waliku...


## Intent Dataset Preparation & Stratified Train/Val/Test Splits

- **What it does:** Loads the project’s synthetic intent-classification dataset containing example Swahili, English, and Sheng/code-switched user requests. The transcripts are normalized using the same text-cleaning pipeline used for the speech transcripts.

- **Why it matters:** Prepares the text data for intent-classification experiments and ensures that the training, validation, and test sets maintain similar proportions of the defined intent classes.

- The dataset is divided into 80% training, 10% validation, and 10% testing subsets using stratified sampling, with random_state=42 for reproducibility.

- The 20% held-out portion is split equally into validation and test sets.

- Exports three stratified datasets: intent_train.csv (80%), intent_val.csv (10%), and intent_test.csv (10%), while preserving the original intent-class proportions across the splits.

In [4]:
# Load synthetic code-switched dataset generated by augment_text.py
synthetic_path = DATA_META_DIR / "synthetic_intent_dataset.csv"
df_intent = pd.read_csv(synthetic_path)

# Normalize transcript text
df_intent["cleaned_transcript"] = df_intent["transcript"].apply(
    clean_swahili_text
)

# Perform 80% Train, 20% Temp (Val + Test) split with stratification
train_df, temp_df = train_test_split(
    df_intent, test_size=0.20, random_state=42, stratify=df_intent["intent"]
)

# Split remaining 20% into equal 10% Validation and 10% Test sets
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["intent"]
)

# Export split datasets to data/processed/
train_df.to_csv(DATA_PROC_DIR / "intent_train.csv", index=False)
val_df.to_csv(DATA_PROC_DIR / "intent_val.csv", index=False)
test_df.to_csv(DATA_PROC_DIR / "intent_test.csv", index=False)

print("Dataset Partition Summary:")
print(f"* Training Set:   {len(train_df)} rows (80%)")
print(f"* Validation Set: {len(val_df)} rows (10%)")
print(f"* Testing Set:    {len(test_df)} rows (10%)")

Dataset Partition Summary:
* Training Set:   800 rows (80%)
* Validation Set: 100 rows (10%)
* Testing Set:    100 rows (10%)


## Class Distribution Verification

- **What it does:** Compares the number of examples belonging to each intent across the training, validation, and test sets.

- **Why it matters:** Verifies that stratified splitting has preserved approximately the same intent-class proportions across the three subsets and helps identify any underlying class imbalance.

In [5]:
# Verify class balance across all splits
split_summary = pd.DataFrame({
    "Train": train_df["intent"].value_counts(),
    "Validation": val_df["intent"].value_counts(),
    "Test": test_df["intent"].value_counts(),
})

print("=== Intent Class Distribution Across Splits ===")
print(split_summary)

=== Intent Class Distribution Across Splits ===
               Train  Validation  Test
intent                                
BUY_AIRTIME      160          20    20
BUY_BUNDLES      160          20    20
CHECK_BALANCE    160          20    20
PAY_BILL         160          20    20
SEND_MONEY       160          20    20


This notebook prepares the datasets for downstream modeling. It first filters invalid speech records and normalizes available transcripts while preserving the original text. It then prepares the synthetic intent-classification dataset and creates stratified 80/10/10 training, validation, and test splits. The split preserves the relative distribution of the defined intent classes across the three subsets, providing a consistent foundation for subsequent model training and evaluation.